In [1]:
import pandas as pd
import numpy as np
from methods import *
import os

In [2]:
# input / output paths
INPUT_PATH = "/home/furkan/projects/cpak/data/ready-to-train/not-prepped"
OUTPUT_PATH = "now-prepped"

In [3]:
def remove_extension(filepath):
    # Separate the extension first
    # Example: "folder/subfolder/image.r.png" -> "folder/subfolder/image.r"
    name_without_ext, _ = os.path.splitext(filepath)
    
    # Replace slashes with underscores to flatten the path
    # Example: "folder/subfolder/image.r" -> "folder_subfolder_image.r"
    flattened_name = name_without_ext.replace("/", "_").replace("\\", "_")
    
    return flattened_name

## Hospital Data Preprocessing
Hospital data needs an additional step for cropping white paddings.

In [4]:
# collect all images under INPUT_PATH (recursively)
valid_extensions = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
image_paths = []
for root, _dirs, files in os.walk(INPUT_PATH):
	for file in files:
		if file.lower().endswith(valid_extensions):
			full_path = os.path.join(root, file)
			# path relative to INPUT_PATH
			image_paths.append(os.path.relpath(full_path, INPUT_PATH))

In [5]:
# create full pipeline
def pipeline(image):
	image = apply_grayscale(image)
	image = apply_crop_white_padding(image, threshold=210)
	# image = apply_fast_resize(image, target_max_dim=1536)
	image = apply_bilateral_filter(image)
	image = apply_clahe(image, clip_limit=3.0)
	image = apply_black_letterbox(image, target_size=(267, 603))
	return image

In [6]:
# process all images
os.makedirs(OUTPUT_PATH, exist_ok=True)
for image_path in image_paths:
	image = load_image(os.path.join(INPUT_PATH, image_path))
	image = pipeline(image)
	out_name = remove_extension(image_path)
	save_ndarray_as_image(image, os.path.join(OUTPUT_PATH, f"{out_name}.png"))